In [13]:
import streamlit as st
import sqlite3
import pandas as pd
st.write(os.getcwd())

NameError: name 'os' is not defined

In [2]:
import sqlite3


class Database:
    def __init__(self):
        self.conn = sqlite3.connect("rehabilitation.db")
        self.cursor = self.conn.cursor()

        # Enable foreign key constraints
        self.cursor.execute("PRAGMA foreign_keys = ON")

        self.create_tables()

    def create_tables(self):

        # ==========================
        # Patients Table
        # ==========================
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS patients(
            patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name TEXT NOT NULL,
            last_name TEXT NOT NULL,
            gender TEXT,
            age INTEGER,
            phone TEXT,
            email TEXT,
            address TEXT,
            addiction TEXT,
            admission_date TEXT,
            therapist_id INTEGER,
            completed_sessions INTEGER DEFAULT 0,
            total_sessions INTEGER DEFAULT 10,

            FOREIGN KEY(therapist_id)
            REFERENCES therapists(therapist_id)
            ON DELETE SET NULL
        )
        """)

        # ==========================
        # Therapists Table
        # ==========================
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS therapists(
            therapist_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name TEXT NOT NULL,
            last_name TEXT NOT NULL,
            specialization TEXT,
            phone TEXT,
            email TEXT
        )
        """)

        # ==========================
        # Appointments Table
        # ==========================
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS appointments(
            appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,

            patient_id INTEGER NOT NULL,

            therapist_id INTEGER NOT NULL,

            appointment_date TEXT NOT NULL,

            appointment_time TEXT NOT NULL,

            status TEXT DEFAULT 'Scheduled',

            FOREIGN KEY(patient_id)
            REFERENCES patients(patient_id)
            ON DELETE CASCADE,

            FOREIGN KEY(therapist_id)
            REFERENCES therapists(therapist_id)
            ON DELETE CASCADE
        )
        """)

        # ==========================
        # Therapy Sessions Table
        # ==========================
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS therapy_sessions(

            session_id INTEGER PRIMARY KEY AUTOINCREMENT,

            patient_id INTEGER,

            therapist_id INTEGER,

            session_date TEXT,

            notes TEXT,

            FOREIGN KEY(patient_id)
            REFERENCES patients(patient_id)
            ON DELETE CASCADE,

            FOREIGN KEY(therapist_id)
            REFERENCES therapists(therapist_id)
            ON DELETE CASCADE
        )
        """)

        self.conn.commit()

    def execute(self, query, values=()):
        self.cursor.execute(query, values)
        self.conn.commit()

    def fetchall(self, query, values=()):
        self.cursor.execute(query, values)
        return self.cursor.fetchall()

    def fetchone(self, query, values=()):
        self.cursor.execute(query, values)
        return self.cursor.fetchone()

    def close(self):
        self.conn.close()


# Create database and tables
if __name__ == "__main__":
    db = Database()
    print("Database and tables created successfully.")
    db.close()

Database and tables created successfully.


In [4]:
#from database import Database


class PatientManager:
    def __init__(self):
        self.db = Database()

    # ==========================
    # Register Patient
    # ==========================
    def register_patient(self):
        try:
            print("\n===== Register Patient =====")

            first_name = input("First Name: ").strip()
            last_name = input("Last Name: ").strip()
            gender = input("Gender: ").strip()
            age = int(input("Age: "))
            phone = input("Phone: ").strip()
            email = input("Email: ").strip()
            address = input("Address: ").strip()
            addiction = input("Addiction Type: ").strip()
            admission_date = input("Admission Date (YYYY-MM-DD): ").strip()

            self.db.execute("""
            INSERT INTO patients
            (first_name,last_name,gender,age,phone,email,address,
            addiction,admission_date)
            VALUES(?,?,?,?,?,?,?,?,?)
            """, (
                first_name,
                last_name,
                gender,
                age,
                phone,
                email,
                address,
                addiction,
                admission_date
            ))

            print("Patient registered successfully.")

        except ValueError:
            print("Invalid input. Age must be a number.")

        except Exception as e:
            print("Error:", e)

    # ==========================
    # View All Patients
    # ==========================
    def view_patients(self):

        patients = self.db.fetchall("""
        SELECT
        patient_id,
        first_name,
        last_name,
        gender,
        age,
        phone,
        addiction,
        admission_date
        FROM patients
        """)

        if not patients:
            print("\nNo patients found.")
            return

        print("\n===== PATIENT LIST =====")

        for patient in patients:
            print(patient)

    # ==========================
    # Search by Patient ID
    # ==========================
    def search_by_id(self):

        try:
            patient_id = int(input("Enter Patient ID: "))

            patient = self.db.fetchone("""
            SELECT * FROM patients
            WHERE patient_id=?
            """, (patient_id,))

            if patient:
                print(patient)
            else:
                print("Patient not found.")

        except ValueError:
            print("Invalid Patient ID.")

    # ==========================
    # Search by Name
    # ==========================
    def search_by_name(self):

        name = input("Enter First Name: ")

        patients = self.db.fetchall("""
        SELECT *
        FROM patients
        WHERE first_name LIKE ?
        """, ('%' + name + '%',))

        if patients:

            for patient in patients:
                print(patient)

        else:
            print("No matching patient.")

    # ==========================
    # Update Patient
    # ==========================
    def update_patient(self):

        try:

            patient_id = int(input("Patient ID: "))

            patient = self.db.fetchone("""
            SELECT * FROM patients
            WHERE patient_id=?
            """, (patient_id,))

            if patient is None:
                print("Patient does not exist.")
                return

            print("\nLeave blank to keep old value.\n")

            phone = input("New Phone: ")
            email = input("New Email: ")
            address = input("New Address: ")

            if phone == "":
                phone = patient[5]

            if email == "":
                email = patient[6]

            if address == "":
                address = patient[7]

            self.db.execute("""
            UPDATE patients
            SET
            phone=?,
            email=?,
            address=?
            WHERE patient_id=?
            """, (
                phone,
                email,
                address,
                patient_id
            ))

            print("Patient updated successfully.")

        except ValueError:
            print("Invalid Patient ID.")

    # ==========================
    # Delete Patient
    # ==========================
    def delete_patient(self):

        try:

            patient_id = int(input("Patient ID: "))

            patient = self.db.fetchone("""
            SELECT *
            FROM patients
            WHERE patient_id=?
            """, (patient_id,))

            if patient is None:
                print("Patient not found.")
                return

            confirm = input(
                "Delete this patient? (Y/N): ").upper()

            if confirm == "Y":

                self.db.execute("""
                DELETE FROM patients
                WHERE patient_id=?
                """, (patient_id,))

                print("Patient deleted successfully.")

            else:
                print("Deletion cancelled.")

        except ValueError:
            print("Invalid Patient ID.")

    # ==========================
    # Assign Therapist
    # ==========================
    def assign_therapist(self):

        try:

            patient_id = int(input("Patient ID: "))
            therapist_id = int(input("Therapist ID: "))

            therapist = self.db.fetchone("""
            SELECT *
            FROM therapists
            WHERE therapist_id=?
            """, (therapist_id,))

            if therapist is None:
                print("Therapist does not exist.")
                return

            self.db.execute("""
            UPDATE patients
            SET therapist_id=?
            WHERE patient_id=?
            """, (
                therapist_id,
                patient_id
            ))

            print("Therapist assigned successfully.")

        except ValueError:
            print("Invalid input.")

    # ==========================
    # Treatment Progress
    # ==========================
    def treatment_progress(self):

        try:

            patient_id = int(input("Patient ID: "))

            result = self.db.fetchone("""
            SELECT
            completed_sessions,
            total_sessions
            FROM patients
            WHERE patient_id=?
            """, (patient_id,))

            if result is None:
                print("Patient not found.")
                return

            completed, total = result

            progress = (completed / total) * 100

            print("\nCompleted Sessions :", completed)
            print("Total Sessions     :", total)
            print(f"Treatment Progress : {progress:.2f}%")

        except ValueError:
            print("Invalid Patient ID.")

    # ==========================
    # Close Database
    # ==========================
    def close(self):
        self.db.close()


# -----------------------------
# Testing
# -----------------------------
if __name__ == "__main__":

    pm = PatientManager()

    while True:

        print("\n====== PATIENT MENU ======")
        print("1. Register Patient")
        print("2. View Patients")
        print("3. Search by ID")
        print("4. Search by Name")
        print("5. Update Patient")
        print("6. Delete Patient")
        print("7. Assign Therapist")
        print("8. View Treatment Progress")
        print("9. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            pm.register_patient()

        elif choice == "2":
            pm.view_patients()

        elif choice == "3":
            pm.search_by_id()

        elif choice == "4":
            pm.search_by_name()

        elif choice == "5":
            pm.update_patient()

        elif choice == "6":
            pm.delete_patient()

        elif choice == "7":
            pm.assign_therapist()

        elif choice == "8":
            pm.treatment_progress()

        elif choice == "9":
            pm.close()
            print("Goodbye!")
            break

        else:
            print("Invalid choice.")


====== PATIENT MENU ======
1. Register Patient
2. View Patients
3. Search by ID
4. Search by Name
5. Update Patient
6. Delete Patient
7. Assign Therapist
8. View Treatment Progress
9. Exit

===== Register Patient =====
Patient registered successfully.

====== PATIENT MENU ======
1. Register Patient
2. View Patients
3. Search by ID
4. Search by Name
5. Update Patient
6. Delete Patient
7. Assign Therapist
8. View Treatment Progress
9. Exit
Invalid choice.

====== PATIENT MENU ======
1. Register Patient
2. View Patients
3. Search by ID
4. Search by Name
5. Update Patient
6. Delete Patient
7. Assign Therapist
8. View Treatment Progress
9. Exit
Goodbye!


In [6]:
#from database import Database


class TherapistManager:

    def __init__(self):
        self.db = Database()

    # =====================================
    # Register Therapist
    # =====================================
    def register_therapist(self):

        try:
            print("\n===== REGISTER THERAPIST =====")

            first_name = input("First Name: ").strip()
            last_name = input("Last Name: ").strip()
            specialization = input("Specialization: ").strip()
            phone = input("Phone: ").strip()
            email = input("Email: ").strip()

            self.db.execute("""
            INSERT INTO therapists
            (first_name,last_name,specialization,phone,email)
            VALUES(?,?,?,?,?)
            """, (
                first_name,
                last_name,
                specialization,
                phone,
                email
            ))

            print("Therapist registered successfully.")

        except Exception as e:
            print("Error:", e)

    # =====================================
    # View Therapists
    # =====================================
    def view_therapists(self):

        therapists = self.db.fetchall("""
        SELECT *
        FROM therapists
        ORDER BY therapist_id
        """)

        if not therapists:
            print("No therapists found.")
            return

        print("\n========== THERAPISTS ==========")

        for therapist in therapists:
            print(therapist)

    # =====================================
    # Search Therapist by ID
    # =====================================
    def search_by_id(self):

        try:

            therapist_id = int(input("Therapist ID: "))

            therapist = self.db.fetchone("""
            SELECT *
            FROM therapists
            WHERE therapist_id=?
            """, (therapist_id,))

            if therapist:
                print(therapist)
            else:
                print("Therapist not found.")

        except ValueError:
            print("Invalid Therapist ID.")

    # =====================================
    # Search Therapist by Name
    # =====================================
    def search_by_name(self):

        name = input("First Name: ")

        therapists = self.db.fetchall("""
        SELECT *
        FROM therapists
        WHERE first_name LIKE ?
        """, ('%' + name + '%',))

        if therapists:

            for therapist in therapists:
                print(therapist)

        else:
            print("No therapist found.")

    # =====================================
    # Update Therapist
    # =====================================
    def update_therapist(self):

        try:

            therapist_id = int(input("Therapist ID: "))

            therapist = self.db.fetchone("""
            SELECT *
            FROM therapists
            WHERE therapist_id=?
            """, (therapist_id,))

            if therapist is None:
                print("Therapist not found.")
                return

            print("\nLeave blank to keep existing value.\n")

            phone = input("New Phone: ").strip()
            email = input("New Email: ").strip()
            specialization = input("New Specialization: ").strip()

            if phone == "":
                phone = therapist[3]

            if email == "":
                email = therapist[4]

            if specialization == "":
                specialization = therapist[2]

            self.db.execute("""
            UPDATE therapists
            SET
            specialization=?,
            phone=?,
            email=?
            WHERE therapist_id=?
            """, (
                specialization,
                phone,
                email,
                therapist_id
            ))

            print("Therapist updated successfully.")

        except ValueError:
            print("Invalid Therapist ID.")

    # =====================================
    # Delete Therapist
    # =====================================
    def delete_therapist(self):

        try:

            therapist_id = int(input("Therapist ID: "))

            therapist = self.db.fetchone("""
            SELECT *
            FROM therapists
            WHERE therapist_id=?
            """, (therapist_id,))

            if therapist is None:
                print("Therapist not found.")
                return

            confirm = input(
                "Delete therapist? (Y/N): ").upper()

            if confirm == "Y":

                self.db.execute("""
                DELETE FROM therapists
                WHERE therapist_id=?
                """, (therapist_id,))

                print("Therapist deleted successfully.")

            else:
                print("Deletion cancelled.")

        except ValueError:
            print("Invalid Therapist ID.")

    # =====================================
    # View Assigned Patients
    # =====================================
    def assigned_patients(self):

        try:

            therapist_id = int(input("Therapist ID: "))

            patients = self.db.fetchall("""
            SELECT
            patient_id,
            first_name,
            last_name,
            addiction,
            admission_date
            FROM patients
            WHERE therapist_id=?
            """, (therapist_id,))

            if not patients:
                print("No patients assigned.")
                return

            print("\nAssigned Patients")

            for patient in patients:
                print(patient)

        except ValueError:
            print("Invalid Therapist ID.")

    def close(self):
        self.db.close()


# =====================================
# Test Menu
# =====================================

if __name__ == "__main__":

    tm = TherapistManager()

    while True:

        print("\n===== THERAPIST MENU =====")
        print("1. Register Therapist")
        print("2. View Therapists")
        print("3. Search Therapist by ID")
        print("4. Search Therapist by Name")
        print("5. Update Therapist")
        print("6. Delete Therapist")
        print("7. View Assigned Patients")
        print("8. Exit")

        choice = input("Choice: ")

        if choice == "1":
            tm.register_therapist()

        elif choice == "2":
            tm.view_therapists()

        elif choice == "3":
            tm.search_by_id()

        elif choice == "4":
            tm.search_by_name()

        elif choice == "5":
            tm.update_therapist()

        elif choice == "6":
            tm.delete_therapist()

        elif choice == "7":
            tm.assigned_patients()

        elif choice == "8":
            tm.close()
            print("Goodbye.")
            break

        else:
            print("Invalid choice.")


===== THERAPIST MENU =====
1. Register Therapist
2. View Therapists
3. Search Therapist by ID
4. Search Therapist by Name
5. Update Therapist
6. Delete Therapist
7. View Assigned Patients
8. Exit

===== REGISTER THERAPIST =====
Therapist registered successfully.

===== THERAPIST MENU =====
1. Register Therapist
2. View Therapists
3. Search Therapist by ID
4. Search Therapist by Name
5. Update Therapist
6. Delete Therapist
7. View Assigned Patients
8. Exit
Goodbye.


In [7]:
#from database import Database


class AppointmentManager:

    def __init__(self):
        self.db = Database()

    # ==========================================
    # Schedule Appointment
    # ==========================================
    def schedule_appointment(self):

        try:
            print("\n===== Schedule Appointment =====")

            patient_id = int(input("Patient ID: "))
            therapist_id = int(input("Therapist ID: "))
            appointment_date = input("Date (YYYY-MM-DD): ").strip()
            appointment_time = input("Time (HH:MM): ").strip()

            # Check patient
            patient = self.db.fetchone(
                "SELECT * FROM patients WHERE patient_id=?",
                (patient_id,)
            )

            if patient is None:
                print("Patient not found.")
                return

            # Check therapist
            therapist = self.db.fetchone(
                "SELECT * FROM therapists WHERE therapist_id=?",
                (therapist_id,)
            )

            if therapist is None:
                print("Therapist not found.")
                return

            # Prevent therapist double booking
            conflict = self.db.fetchone("""
            SELECT *
            FROM appointments
            WHERE therapist_id=?
            AND appointment_date=?
            AND appointment_time=?
            AND status='Scheduled'
            """, (
                therapist_id,
                appointment_date,
                appointment_time
            ))

            if conflict:
                print("Therapist already has an appointment at that time.")
                return

            self.db.execute("""
            INSERT INTO appointments
            (patient_id,
            therapist_id,
            appointment_date,
            appointment_time)
            VALUES(?,?,?,?)
            """, (
                patient_id,
                therapist_id,
                appointment_date,
                appointment_time
            ))

            print("Appointment scheduled successfully.")

        except ValueError:
            print("Invalid input.")

    # ==========================================
    # View Appointments
    # ==========================================
    def view_appointments(self):

        appointments = self.db.fetchall("""

        SELECT

        appointments.appointment_id,

        patients.first_name || ' ' || patients.last_name,

        therapists.first_name || ' ' || therapists.last_name,

        appointments.appointment_date,

        appointments.appointment_time,

        appointments.status

        FROM appointments

        JOIN patients
        ON appointments.patient_id = patients.patient_id

        JOIN therapists
        ON appointments.therapist_id = therapists.therapist_id

        ORDER BY appointments.appointment_date

        """)

        if not appointments:
            print("No appointments found.")
            return

        print("\n===== APPOINTMENTS =====")

        for appointment in appointments:
            print(appointment)

    # ==========================================
    # Cancel Appointment
    # ==========================================
    def cancel_appointment(self):

        try:

            appointment_id = int(input("Appointment ID: "))

            appointment = self.db.fetchone("""
            SELECT *
            FROM appointments
            WHERE appointment_id=?
            """, (appointment_id,))

            if appointment is None:
                print("Appointment not found.")
                return

            self.db.execute("""
            UPDATE appointments
            SET status='Cancelled'
            WHERE appointment_id=?
            """, (appointment_id,))

            print("Appointment cancelled.")

        except ValueError:
            print("Invalid appointment ID.")

    # ==========================================
    # Reschedule Appointment
    # ==========================================
    def reschedule_appointment(self):

        try:

            appointment_id = int(input("Appointment ID: "))

            new_date = input("New Date (YYYY-MM-DD): ").strip()
            new_time = input("New Time (HH:MM): ").strip()

            appointment = self.db.fetchone("""
            SELECT therapist_id
            FROM appointments
            WHERE appointment_id=?
            """, (appointment_id,))

            if appointment is None:
                print("Appointment not found.")
                return

            therapist_id = appointment[0]

            conflict = self.db.fetchone("""
            SELECT *
            FROM appointments
            WHERE therapist_id=?
            AND appointment_date=?
            AND appointment_time=?
            AND status='Scheduled'
            """, (
                therapist_id,
                new_date,
                new_time
            ))

            if conflict:
                print("Therapist is already booked.")
                return

            self.db.execute("""
            UPDATE appointments
            SET
            appointment_date=?,
            appointment_time=?
            WHERE appointment_id=?
            """, (
                new_date,
                new_time,
                appointment_id
            ))

            print("Appointment rescheduled.")

        except ValueError:
            print("Invalid input.")

    # ==========================================
    # Upcoming Appointments
    # ==========================================
    def upcoming_appointments(self):

        appointments = self.db.fetchall("""

        SELECT

        appointment_id,

        patient_id,

        therapist_id,

        appointment_date,

        appointment_time,

        status

        FROM appointments

        WHERE status='Scheduled'

        ORDER BY appointment_date,
        appointment_time

        """)

        if not appointments:
            print("No upcoming appointments.")
            return

        print("\n===== UPCOMING APPOINTMENTS =====")

        for appointment in appointments:
            print(appointment)

    # ==========================================
    # Patient Appointment History
    # ==========================================
    def patient_history(self):

        try:

            patient_id = int(input("Patient ID: "))

            appointments = self.db.fetchall("""

            SELECT

            appointment_date,

            appointment_time,

            status

            FROM appointments

            WHERE patient_id=?

            ORDER BY appointment_date

            """, (patient_id,))

            if not appointments:
                print("No appointment history.")
                return

            print("\nAppointment History")

            for appointment in appointments:
                print(appointment)

        except ValueError:
            print("Invalid Patient ID.")

    def close(self):
        self.db.close()


# ==========================================
# Test Menu
# ==========================================

if __name__ == "__main__":

    am = AppointmentManager()

    while True:

        print("\n===== APPOINTMENT MENU =====")
        print("1. Schedule Appointment")
        print("2. View Appointments")
        print("3. Cancel Appointment")
        print("4. Reschedule Appointment")
        print("5. Upcoming Appointments")
        print("6. Patient Appointment History")
        print("7. Exit")

        choice = input("Choice: ")

        if choice == "1":
            am.schedule_appointment()

        elif choice == "2":
            am.view_appointments()

        elif choice == "3":
            am.cancel_appointment()

        elif choice == "4":
            am.reschedule_appointment()

        elif choice == "5":
            am.upcoming_appointments()

        elif choice == "6":
            am.patient_history()

        elif choice == "7":
            am.close()
            print("Goodbye.")
            break

        else:
            print("Invalid menu choice.")


===== APPOINTMENT MENU =====
1. Schedule Appointment
2. View Appointments
3. Cancel Appointment
4. Reschedule Appointment
5. Upcoming Appointments
6. Patient Appointment History
7. Exit

===== Schedule Appointment =====
Appointment scheduled successfully.

===== APPOINTMENT MENU =====
1. Schedule Appointment
2. View Appointments
3. Cancel Appointment
4. Reschedule Appointment
5. Upcoming Appointments
6. Patient Appointment History
7. Exit
Goodbye.


In [9]:
#from database import Database


class ProgressManager:

    def __init__(self):
        self.db = Database()

    # ==========================================
    # Record Therapy Session
    # ==========================================
    def record_session(self):

        try:

            print("\n===== RECORD THERAPY SESSION =====")

            patient_id = int(input("Patient ID: "))
            therapist_id = int(input("Therapist ID: "))
            session_date = input("Session Date (YYYY-MM-DD): ")
            notes = input("Session Notes: ")

            # Verify patient exists
            patient = self.db.fetchone(
                "SELECT completed_sessions, total_sessions FROM patients WHERE patient_id=?",
                (patient_id,)
            )

            if patient is None:
                print("Patient not found.")
                return

            # Verify therapist exists
            therapist = self.db.fetchone(
                "SELECT therapist_id FROM therapists WHERE therapist_id=?",
                (therapist_id,)
            )

            if therapist is None:
                print("Therapist not found.")
                return

            completed, total = patient

            if completed >= total:
                print("Treatment has already been completed.")
                return

            # Save therapy session
            self.db.execute("""
            INSERT INTO therapy_sessions
            (patient_id, therapist_id, session_date, notes)
            VALUES (?,?,?,?)
            """, (
                patient_id,
                therapist_id,
                session_date,
                notes
            ))

            # Increment completed sessions
            self.db.execute("""
            UPDATE patients
            SET completed_sessions = completed_sessions + 1
            WHERE patient_id=?
            """, (patient_id,))

            print("Therapy session recorded successfully.")

        except ValueError:
            print("Invalid input.")

    # ==========================================
    # Calculate Progress
    # ==========================================
    def calculate_progress(self):

        try:

            patient_id = int(input("Patient ID: "))

            patient = self.db.fetchone("""
            SELECT
            first_name,
            last_name,
            completed_sessions,
            total_sessions
            FROM patients
            WHERE patient_id=?
            """, (patient_id,))

            if patient is None:
                print("Patient not found.")
                return

            first, last, completed, total = patient

            percentage = (completed / total) * 100

            print("\n===== TREATMENT PROGRESS =====")
            print("Patient :", first, last)
            print("Completed Sessions :", completed)
            print("Total Sessions     :", total)
            print(f"Progress            : {percentage:.2f}%")

            if percentage == 100:
                print("Treatment Completed Successfully!")

        except ValueError:
            print("Invalid Patient ID.")

    # ==========================================
    # View Therapy Session History
    # ==========================================
    def session_history(self):

        try:

            patient_id = int(input("Patient ID: "))

            sessions = self.db.fetchall("""

            SELECT

            session_id,

            session_date,

            notes

            FROM therapy_sessions

            WHERE patient_id=?

            ORDER BY session_date

            """, (patient_id,))

            if not sessions:
                print("No therapy sessions found.")
                return

            print("\n===== THERAPY SESSION HISTORY =====")

            for session in sessions:
                print(session)

        except ValueError:
            print("Invalid Patient ID.")

    # ==========================================
    # Patient Progress Report
    # ==========================================
    def progress_report(self):

        report = self.db.fetchall("""

        SELECT

        patient_id,

        first_name,

        last_name,

        addiction,

        completed_sessions,

        total_sessions

        FROM patients

        ORDER BY patient_id

        """)

        if not report:
            print("No patient records found.")
            return

        print("\n========== PATIENT PROGRESS REPORT ==========")

        for row in report:

            patient_id, first, last, addiction, completed, total = row

            progress = (completed / total) * 100

            print("------------------------------------------")
            print("Patient ID :", patient_id)
            print("Name       :", first, last)
            print("Addiction  :", addiction)
            print("Completed  :", completed)
            print("Total      :", total)
            print(f"Progress   : {progress:.1f}%")

    # ==========================================
    # Update Total Sessions
    # ==========================================
    def update_total_sessions(self):

        try:

            patient_id = int(input("Patient ID: "))
            total = int(input("New Total Sessions: "))

            if total <= 0:
                print("Total sessions must be greater than zero.")
                return

            self.db.execute("""
            UPDATE patients
            SET total_sessions=?
            WHERE patient_id=?
            """, (
                total,
                patient_id
            ))

            print("Total sessions updated.")

        except ValueError:
            print("Invalid input.")

    def close(self):
        self.db.close()


# ==========================================
# Testing Menu
# ==========================================

if __name__ == "__main__":

    pm = ProgressManager()

    while True:

        print("\n===== PROGRESS MENU =====")
        print("1. Record Therapy Session")
        print("2. Calculate Treatment Progress")
        print("3. View Therapy Session History")
        print("4. Patient Progress Report")
        print("5. Update Total Sessions")
        print("6. Exit")

        choice = input("Choice: ")

        if choice == "1":
            pm.record_session()

        elif choice == "2":
            pm.calculate_progress()

        elif choice == "3":
            pm.session_history()

        elif choice == "4":
            pm.progress_report()

        elif choice == "5":
            pm.update_total_sessions()

        elif choice == "6":
            pm.close()
            print("Goodbye.")
            break

        else:
            print("Invalid menu choice.")


===== PROGRESS MENU =====
1. Record Therapy Session
2. Calculate Treatment Progress
3. View Therapy Session History
4. Patient Progress Report
5. Update Total Sessions
6. Exit



===== RECORD THERAPY SESSION =====
Therapy session recorded successfully.

===== PROGRESS MENU =====
1. Record Therapy Session
2. Calculate Treatment Progress
3. View Therapy Session History
4. Patient Progress Report
5. Update Total Sessions
6. Exit
Goodbye.


In [11]:
#from patient import PatientManager
#from therapist import TherapistManager
#from appointment import AppointmentManager
#from progress import ProgressManager


patient = PatientManager()
therapist = TherapistManager()
appointment = AppointmentManager()
progress = ProgressManager()


def patient_menu():

    while True:

        print("\n========== PATIENT MANAGEMENT ==========")
        print("1. Register Patient")
        print("2. View Patients")
        print("3. Search Patient by ID")
        print("4. Search Patient by Name")
        print("5. Update Patient")
        print("6. Delete Patient")
        print("7. Assign Therapist")
        print("8. View Treatment Progress")
        print("9. Back")

        choice = input("Enter choice: ")

        if choice == "1":
            patient.register_patient()

        elif choice == "2":
            patient.view_patients()

        elif choice == "3":
            patient.search_by_id()

        elif choice == "4":
            patient.search_by_name()

        elif choice == "5":
            patient.update_patient()

        elif choice == "6":
            patient.delete_patient()

        elif choice == "7":
            patient.assign_therapist()

        elif choice == "8":
            patient.treatment_progress()

        elif choice == "9":
            break

        else:
            print("Invalid choice.")


def therapist_menu():

    while True:

        print("\n========== THERAPIST MANAGEMENT ==========")
        print("1. Register Therapist")
        print("2. View Therapists")
        print("3. Search Therapist by ID")
        print("4. Search Therapist by Name")
        print("5. Update Therapist")
        print("6. Delete Therapist")
        print("7. View Assigned Patients")
        print("8. Back")

        choice = input("Enter choice: ")

        if choice == "1":
            therapist.register_therapist()

        elif choice == "2":
            therapist.view_therapists()

        elif choice == "3":
            therapist.search_by_id()

        elif choice == "4":
            therapist.search_by_name()

        elif choice == "5":
            therapist.update_therapist()

        elif choice == "6":
            therapist.delete_therapist()

        elif choice == "7":
            therapist.assigned_patients()

        elif choice == "8":
            break

        else:
            print("Invalid choice.")


def appointment_menu():

    while True:

        print("\n========== APPOINTMENT MANAGEMENT ==========")
        print("1. Schedule Appointment")
        print("2. View Appointments")
        print("3. Cancel Appointment")
        print("4. Reschedule Appointment")
        print("5. View Upcoming Appointments")
        print("6. View Patient Appointment History")
        print("7. Back")

        choice = input("Enter choice: ")

        if choice == "1":
            appointment.schedule_appointment()

        elif choice == "2":
            appointment.view_appointments()

        elif choice == "3":
            appointment.cancel_appointment()

        elif choice == "4":
            appointment.reschedule_appointment()

        elif choice == "5":
            appointment.upcoming_appointments()

        elif choice == "6":
            appointment.patient_history()

        elif choice == "7":
            break

        else:
            print("Invalid choice.")


def progress_menu():

    while True:

        print("\n========== THERAPY PROGRESS ==========")
        print("1. Record Therapy Session")
        print("2. Calculate Treatment Progress")
        print("3. View Therapy Session History")
        print("4. Patient Progress Report")
        print("5. Update Total Sessions")
        print("6. Back")

        choice = input("Enter choice: ")

        if choice == "1":
            progress.record_session()

        elif choice == "2":
            progress.calculate_progress()

        elif choice == "3":
            progress.session_history()

        elif choice == "4":
            progress.progress_report()

        elif choice == "5":
            progress.update_total_sessions()

        elif choice == "6":
            break

        else:
            print("Invalid choice.")


# ===============================
# Main Program
# ===============================

while True:

    print("\n========================================")
    print(" REHABILITATION MANAGEMENT SYSTEM ")
    print("========================================")

    print("1. Patient Management")
    print("2. Therapist Management")
    print("3. Appointment Management")
    print("4. Therapy Progress")
    print("5. Exit")

    option = input("Select Option: ")

    if option == "1":
        patient_menu()

    elif option == "2":
        therapist_menu()

    elif option == "3":
        appointment_menu()

    elif option == "4":
        progress_menu()

    elif option == "5":

        patient.close()
        therapist.close()
        appointment.close()
        progress.close()

        print("Thank you for using the Rehabilitation Management System.")
        break

    else:
        print("Invalid option. Please try again.")


 REHABILITATION MANAGEMENT SYSTEM 
1. Patient Management
2. Therapist Management
3. Appointment Management
4. Therapy Progress
5. Exit



========== THERAPIST MANAGEMENT ==========
1. Register Therapist
2. View Therapists
3. Search Therapist by ID
4. Search Therapist by Name
5. Update Therapist
6. Delete Therapist
7. View Assigned Patients
8. Back

Leave blank to keep existing value.

Therapist updated successfully.

========== THERAPIST MANAGEMENT ==========
1. Register Therapist
2. View Therapists
3. Search Therapist by ID
4. Search Therapist by Name
5. Update Therapist
6. Delete Therapist
7. View Assigned Patients
8. Back
Invalid choice.

========== THERAPIST MANAGEMENT ==========
1. Register Therapist
2. View Therapists
3. Search Therapist by ID
4. Search Therapist by Name
5. Update Therapist
6. Delete Therapist
7. View Assigned Patients
8. Back

 REHABILITATION MANAGEMENT SYSTEM 
1. Patient Management
2. Therapist Management
3. Appointment Management
4. Therapy Progress
5. Exit
Invalid option. Please try again.

 REHABILITATION MANAGEMENT SYSTEM 
1. Patient Management
2. Therapist Management
3. Appointment Manageme